# Sample Code to Run NN-IC-Log

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import random
import torch

seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

## Load Data

In [3]:
# Prerequisite: pip install SurvSet
from SurvSet.data import SurvLoader

dataset_name = "breast"
loader = SurvLoader()
df, _ = loader.load_dataset(ds_name=dataset_name).values()
print(df.head())

   pid  event       time fac_TumorStage fac_NodalStatus fac_Histology  \
0    0      1   1.546053              1               0             1   
1    1      1   3.519737              0               0             1   
2    2      1   8.059211              1               1             1   
3    3      1   8.651316              1               0             1   
4    4      1  12.138158              1               1             1   

  fac_CathepsinD  
0              0  
1              1  
2              1  
3              1  
4              0  


## Encode Features

In [4]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# rescale numerical features into [0, 1]
cols_cont = [col for col in df.columns if col.startswith("num_")]
if len(cols_cont) > 0:
    df[cols_cont] = df[cols_cont].fillna(df[cols_cont].median())
    mm = MinMaxScaler()
    df[cols_cont] = mm.fit_transform(df[cols_cont])

# encode categorical columns
cols_cat = []
for col in df.columns:
    if col.startswith("fac_"):
        cols_cat.append(col)
df = pd.get_dummies(df, columns=cols_cat, drop_first=True, dtype="float32")
print(df.head())

   pid  event       time  fac_TumorStage_1  fac_NodalStatus_1  \
0    0      1   1.546053               1.0                0.0   
1    1      1   3.519737               0.0                0.0   
2    2      1   8.059211               1.0                1.0   
3    3      1   8.651316               1.0                0.0   
4    4      1  12.138158               1.0                1.0   

   fac_Histology_1  fac_CathepsinD_1  
0              1.0               0.0  
1              1.0               1.0  
2              1.0               1.0  
3              1.0               1.0  
4              1.0               0.0  


## Set Target Values

In [5]:
mask_exact = df["event"] == 1
lb = df["time"].values.copy()
ub = df["time"].values.copy()
lb[mask_exact] -= 0.00001
ub[~mask_exact] = np.inf

df['lb'] = lb
df['ub'] = ub
print(df)

    pid  event       time  fac_TumorStage_1  fac_NodalStatus_1  \
0     0      1   1.546053               1.0                0.0   
1     1      1   3.519737               0.0                0.0   
2     2      1   8.059211               1.0                1.0   
3     3      1   8.651316               1.0                0.0   
4     4      1  12.138158               1.0                1.0   
..  ...    ...        ...               ...                ...   
95   95      0  72.000000               1.0                0.0   
96   96      0  72.000000               0.0                0.0   
97   97      0  72.000000               0.0                0.0   
98   98      0  72.000000               0.0                0.0   
99   99      0  72.000000               0.0                1.0   

    fac_Histology_1  fac_CathepsinD_1         lb         ub  
0               1.0               0.0   1.546043   1.546053  
1               1.0               1.0   3.519727   3.519737  
2               1.0  

## Split Dataset

In [6]:
from sklearn.model_selection import train_test_split

min_y = 0.0
max_y = df["time"].max()

df_train, df_test = train_test_split(df, random_state=42, test_size=0.2)
print(df_train.shape, df_test.shape)

(80, 9) (20, 9)


## Train with Neural Network (NN-IC-Log)

In [7]:
from cenreg.pytorch.datamodule import IntervalCensoredDataModule
from cenreg.pytorch.loss_cdf import NegativeLogLikelihoodInterval
from cenreg.pytorch.mlp import MLP

# drop the columns that are not needed
x_train = df_train.drop(columns=["pid", "time", "event", "lb", "ub"]).values
x_test = df_test.drop(columns=["pid", "time", "event", "lb", "ub"]).values

# preparation for PyTorch training
dm = IntervalCensoredDataModule(batch_size=128)
lb_train = df_train["lb"].values
ub_train = df_train["ub"].values
train_dataloader = dm.train_dataloader(x_train, lb_train, ub_train)
y_bins = np.unique([lb_train, ub_train])
if y_bins[0] == -np.inf:
    y_bins = y_bins[1:]
if y_bins[0] > 0.0:
    y_bins = np.append([0.0], y_bins)
if y_bins[-1] == np.inf:
    y_bins = y_bins[:-1]
num_bins = len(y_bins) - 1
model = MLP(x_train.shape[1], num_bins, 32)

# prepare the loss function
loss_fn = NegativeLogLikelihoodInterval(torch.tensor(y_bins, dtype=torch.float32))

In [8]:
#from cenreg.pytorch.distribution import LinearCDF

optimizer_mlp = torch.optim.Adam(model.parameters(), lr=1.0)
for epoch in range(10):
    loss_sum = 0.0
    for i, data in enumerate(train_dataloader):
        x, lb, ub = data

        optimizer_mlp.zero_grad()
        pred = model(x)
        loss = loss_fn.loss(pred, lb, ub).mean()
        loss.backward()
        loss_sum += loss.item()
        optimizer_mlp.step()
    print(f"Epoch {epoch+1}, Train Loss: {loss_sum / len(train_dataloader)}")

Epoch 1, Train Loss: 3.4342989921569824
Epoch 2, Train Loss: 2.3025100231170654
Epoch 3, Train Loss: 2.3025097846984863
Epoch 4, Train Loss: 2.3025102615356445
Epoch 5, Train Loss: 2.3025102615356445
Epoch 6, Train Loss: 2.3025100231170654
Epoch 7, Train Loss: 2.3025100231170654
Epoch 8, Train Loss: 2.3025097846984863
Epoch 9, Train Loss: 2.3025097846984863
Epoch 10, Train Loss: 2.3025097846984863


## Predict

In [9]:
from cenreg.distribution.cdf import CumulativeDist

lb_test = df_test["lb"].values
ub_test = df_test["ub"].values
test_dataloader = dm.test_dataloader(x_test, lb_test, ub_test)

model.eval()
with torch.no_grad():
    for i, data in enumerate(test_dataloader):
        x, _, _ = data
        pred = model(x)
        break  # assuming that the first batch contains all the test data

dist = CumulativeDist(y_bins, cum_p=pred.detach().cpu().numpy())

## Evaluate Model

In [10]:
import cenreg.metric.cdf
import cenreg.metric.quantile

# Compute Logarithmic score on CDF representation
icnll = cenreg.metric.cdf.negative_log_likelihood_interval(dist, lb_test, ub_test)
print("IC-NLL", icnll.mean())

# Compute IC-Log calibration error on CDF representation
ic_cal = cenreg.metric.quantile.ic_calibration(dist, lb_test, ub_test)
print("IC-Cal", ic_cal)

IC-NLL 2.7630321150926216
IC-Cal 0.053333333333333344
